<a href="https://colab.research.google.com/github/mauro-nievoff/MultiCaRe_Dataset/blob/main/1_How_to_Query_Case_Reports_from_PubMed_using_BioPython.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# How to Query Case Reports from PubMed using Biopython (Bio.Entrez)

When it comes to biomedical literature, [PubMed](https://pubmed.ncbi.nlm.nih.gov/) is the most comprehensive database, but querying data from it can sometimes be a complex task. In this notebook we will delve into the use of [Biopython](https://biopython.org/), a versatile bioinformatic toolkit to search and retrieve relevant open-access articles from PubMed.

We will discuss:
- How to create a search string
- Retrieving article IDs
- Tips to run a large query

## 📓 General Concepts about PubMed Searches

Generally speaking, a PubMed search should be as specific and sensitive as possible. This means that it should return as many relevant articles while minimizing the inclusion of irrelevant ones.

Let's see an example. If we wanted to get all the case reports published in 2022 related to hypertension, we would write something like this:

```('high blood pressure'[Title/Abstract] OR 'Hyptertension'[Mesh]) AND case reports[Publication Type] AND 2022[Date - Publication]```

This search string includes:
- pertinent strings (using quotation marks if the match should be exact), such as `high blood pressure` or `2022`
- the field in which they should be searched (between square brackets), such as `[Mesh]` or `[Publication Type]`
- boolean operators (using brackets as necessary), such as `AND` or `OR`

In order to get the most appropriate string for a specific use case, an iterative process should be followed: first running a query, then analyzing the results and finally modifying the search string if necessary. This is the kind of questions that should be kept in mind while refining the search:
- Is it necessary to add a new filter?
- Is there any other relevant term that should be included in the string?
- Is there any specific term associated with irrelevant articles that should be used together with the boolean operator `NOT`?

## 🔍 Creating a Search String for Case Report Articles

The create_search_string() function is used to create a search string to retrieve as many relevant case reports as possible given a specific clinical use case (e.g. a specific disease, symptom or therapy). When no use case is provided, the generic string 'case' is used, in order to get any case report.

In [90]:
def create_search_string(clinical_usecase=""):
    if not clinical_usecase:
        raise ValueError("clinical_usecase must specify a disease or condition.")

    disease_search = f"{clinical_usecase}[Title/Abstract]"

    case_report_filter = '"case reports"[Publication Type]'

    full_text_filter = "free full text[sb]"

    animal_exclusion = " NOT animal[Filter]"

    search_string = (
        f"{disease_search} AND "
        f"{case_report_filter} AND "
        f"{full_text_filter}"
        f"{animal_exclusion}"

    )

    return search_string
    # return disease_search

In [91]:
search_string = create_search_string("Malaria")

In [92]:
search_string

'Malaria[Title/Abstract] AND "case reports"[Publication Type] AND free full text[sb] NOT animal[Filter]'

In [8]:
%%capture
!pip install Bio

In [52]:
from Bio import Entrez

In order to use Entrez, the email address from an NCBI account should be set, along with an API key (available at https://www.ncbi.nlm.nih.gov/account/settings/).

In [76]:
Entrez.email = "trangiahuan2411@gmail.com"
Entrez.api_key = "ef15e8a70ab9b8a2b0c79573c3c9d045fb08"

In [93]:
print("Search string:", search_string)

handle = Entrez.esearch(
    db="pubmed",
    term=search_string,
    retmode="xml",
    retmax=10000
)
record = Entrez.read(handle)
handle.close()

print("Total matching records:", record["Count"])
print("PMIDs returned:", len(record["IdList"]))
print("Query translation:", record.get("QueryTranslation"))
print("Warnings:", record.get("WarningList"))

Search string: Malaria[Title/Abstract] AND "case reports"[Publication Type] AND free full text[sb] NOT animal[Filter]
Total matching records: 1304
PMIDs returned: 1304
Query translation: ("Malaria"[Title/Abstract] AND "case reports"[Publication Type] AND "loattrfree full text"[Filter]) NOT "animals"[MeSH Terms:noexp]
Warnings: {'QuotedPhraseNotFound': [], 'PhraseIgnored': [], 'OutputMessage': ['Restrictions achieved. start and count adjusted to 0, 9999']}


In [36]:
print(f"A total amount of {len(pmid_list)} were retrieved.")

A total amount of 86 were retrieved.


PMIDs can be easily mapped to PubMed Central IDs (PMCIDs) with the function below.

In [13]:
def get_pmcid(pmid):
  pmid_handle = Entrez.efetch(db="pubmed", id=pmid, rettype="xml")
  pmid_record = Entrez.read(pmid_handle)
  article_ids = pmid_record['PubmedArticle'][0]['PubmedData']['ArticleIdList']
  for e in article_ids:
    if e.attributes['IdType'] == 'pmc':
      pmcid = str(e)
      break
    else:
      pmcid = 'not_found'
  return pmcid

In [14]:
def get_pmids(search_string, start_year = 2005, end_year = 2025):

  pmid_list = []

  for year in range(int(start_year), int(end_year)+1):
    date_string = str(year)
    query = search_string + ' AND ' + date_string + '[Date - Publication]'
    handle = Entrez.esearch(db="pubmed", term=query, retmode="xml", retmax= 10000)
    record = Entrez.read(handle)
    pmid_list += record["IdList"]

  pmid_list = list(set(pmid_list))
  return pmid_list

In [15]:
full_pmid_list = get_pmids(search_string)

In [29]:
print(f"A total amount of {len(full_pmid_list)} were retrieved.")

A total amount of 1954 were retrieved.


In [30]:
print(full_pmid_list)

['20074103', '29915366', '29764474', '30911529', '29242073', '34730691', '37867542', '39255515', '21291597', '27514973', '38059153', '21708032', '17283643', '40214319', '34306776', '20693320', '38938742', '36284305', '34303353', '36645165', '31818297', '23608386', '31080736', '17900210', '19356225', '40056919', '23358640', '20962732', '20875291', '31563918', '19742268', '30683044', '30115637', '23276745', '21603649', '24489564', '36844613', '20429877', '21853089', '24315069', '30199554', '39006507', '26894117', '33688269', '31595871', '22414042', '28784903', '27279973', '16369313', '23754912', '17687849', '39183741', '32172256', '40064630', '31296177', '26435988', '28732806', '22000366', '23530968', '38814096', '22185638', '27642407', '26793463', '38556861', '18057166', '33730099', '26417148', '22898486', '24972637', '34306101', '36632535', '32746919', '34457209', '25567973', '36439402', '29070083', '23109969', '36297130', '27757247', '30518411', '30952502', '32636981', '26240154', '29

If we wanted to get even more PMIDs, then more granular temporal filters could be used (by month or by day).